# Document Restoration Pipeline - Dataset Exploration

Based on research by **Cosmas Kiptoo Sang**
Kisii University, March 2026

This notebook explores the scanned document dataset and demonstrates the restoration pipeline.

In [ ]:
# Import libraries
import sys
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import json
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = Path('..').resolve()
sys.path.append(str(project_root))

# Import project modules
from config import *
from src.denoising import ImageDenoiser, load_image, compute_psnr, compute_ssim
from src.deskewing import DeskewPipeline
from src.ocr_module import OCRProcessor

## 1. Dataset Overview

In [ ]:
# Load dataset information
print("Dataset Structure:")
print(f"Project Root: {project_root}")
print(f"Data Directory: {DATA_DIR}")
print(f"Train Images: {TRAIN_IMAGES_DIR}")
print(f"Test Images: {TEST_IMAGES_DIR}")
print(f"Train Labels: {TRAIN_LABELS_DIR}")

# Count files
train_images = list(TRAIN_IMAGES_DIR.glob("*.png"))
train_labels = list(TRAIN_LABELS_DIR.glob("*.txt"))

print(f"\nFile Counts:")
print(f"Training Images: {len(train_images)}")
print(f"Training Labels: {len(train_labels)}")

# Load train/test lists
with open(TRAIN_LIST_PATH, 'r') as f:
    train_list = json.load(f)
    
with open(TEST_LIST_PATH, 'r') as f:
    test_list = json.load(f)
    
print(f"\nTrain List: {len(train_list)} images")
print(f"Test List: {len(test_list)} images")

## 2. Sample Image Analysis

In [ ]:
# Load and display sample images
def display_sample_images(n_samples=5):
    fig, axes = plt.subplots(1, n_samples, figsize=(20, 4))
    
    for i, img_name in enumerate(train_list[:n_samples]):
        img_path = TRAIN_IMAGES_DIR / img_name
        if img_path.exists():
            img = cv2.imread(str(img_path))
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            axes[i].imshow(img_rgb)
            axes[i].set_title(f"{img_name}\nShape: {img.shape}")
            axes[i].axis('off')
            
            # Load angle label if available
            label_path = TRAIN_LABELS_DIR / f"{img_name.replace('.png', '.txt')}"
            if label_path.exists():
                with open(label_path, 'r') as f:
                    angle = float(f.read().strip())
                axes[i].set_xlabel(f"Angle: {angle:.2f}°")
    
    plt.tight_layout()
    plt.show()

display_sample_images(5)

## 3. Label Distribution Analysis

In [ ]:
# Analyze angle labels
def analyze_labels():
    angles = []
    
    for label_file in tqdm(train_labels[:100], desc="Loading labels"):
        try:
            with open(label_file, 'r') as f:
                angle = float(f.read().strip())
                angles.append(angle)
        except:
            continue
    
    if angles:
        angles = np.array(angles)
        
        print(f"Label Statistics (n={len(angles)}):")
        print(f"  Mean: {np.mean(angles):.3f}°")
        print(f"  Std: {np.std(angles):.3f}°")
        print(f"  Min: {np.min(angles):.3f}°")
        print(f"  Max: {np.max(angles):.3f}°")
        print(f"  Median: {np.median(angles):.3f}°")
        
        # Plot distribution
        plt.figure(figsize=(10, 6))
        plt.hist(angles, bins=30, edgecolor='black', alpha=0.7)
        plt.xlabel('Rotation Angle (degrees)')
        plt.ylabel('Frequency')
        plt.title('Distribution of Document Rotation Angles')
        plt.grid(True, alpha=0.3)
        plt.show()
        
        return angles
    else:
        print("No labels found!")
        return None

angles = analyze_labels()

## 4. Denoising Demonstration

In [ ]:
# Test denoising methods on a sample image
def test_denoising():
    # Load a sample image
    sample_img_path = TRAIN_IMAGES_DIR / train_list[0]
    if not sample_img_path.exists():
        print(f"Sample image not found: {sample_img_path}")
        return
    
    image = load_image(sample_img_path)
    print(f"Original Image: {image.shape}, dtype: {image.dtype}")
    
    # Test different denoising methods
    denoiser = ImageDenoiser()
    methods = [
        "gaussian_blur",
        "median_filter",
        "bilateral_filter",
        "non_local_means",
        "morphological_opening",
        "clahe"
    ]
    
    # Create figure
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    # Display original
    axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original Image")
    axes[0].axis('off')
    
    # Apply and display each method
    for i, method in enumerate(methods, 1):
        try:
            denoised = denoiser.denoise(image, method)
            axes[i].imshow(cv2.cvtColor(denoised, cv2.COLOR_BGR2RGB))
            
            # Compute metrics
            psnr = compute_psnr(image, denoised)
            ssim = compute_ssim(image, denoised)
            
            axes[i].set_title(f"{method}\nPSNR: {psnr:.1f} dB, SSIM: {ssim:.3f}")
            axes[i].axis('off')
            
        except Exception as e:
            axes[i].text(0.5, 0.5, f"Error:\n{str(e)}", 
                        ha='center', va='center', transform=axes[i].transAxes)
            axes[i].set_title(method)
            axes[i].axis('off')
    
    # Hide unused subplot
    axes[7].axis('off')
    
    plt.tight_layout()
    plt.show()

test_denoising()

## 5. Deskewing Demonstration

In [ ]:
# Test deskewing on a rotated image
def test_deskewing():
    # Load a sample image
    sample_img_path = TRAIN_IMAGES_DIR / train_list[0]
    if not sample_img_path.exists():
        print(f"Sample image not found: {sample_img_path}")
        return
    
    image = load_image(sample_img_path)
    
    # Create a rotated version for testing
    angle = 15  # degrees
    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, rotation_matrix, (w, h))
    
    # Initialize deskewing pipeline
    # Note: This requires a trained model
    try:
        deskewer = DeskewPipeline()
        deskewed, predicted_angle = deskewer.deskew_image(rotated)
        
        # Display results
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        axes[0].set_title("Original Image")
        axes[0].axis('off')
        
        axes[1].imshow(cv2.cvtColor(rotated, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f"Rotated by {angle}°")
        axes[1].axis('off')
        
        axes[2].imshow(cv2.cvtColor(deskewed, cv2.COLOR_BGR2RGB))
        axes[2].set_title(f"Deskewed\nPredicted Angle: {predicted_angle:.2f}°")
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Deskewing test failed (model may not be trained): {e}")
        
        # Show rotated image only
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        axes[0].set_title("Original Image")
        axes[0].axis('off')
        
        axes[1].imshow(cv2.cvtColor(rotated, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f"Rotated by {angle}° (for testing)")
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()

test_deskewing()

## 6. OCR Demonstration

In [ ]:
# Test OCR on a sample image
def test_ocr():
    # Load a sample image
    sample_img_path = TRAIN_IMAGES_DIR / train_list[0]
    if not sample_img_path.exists():
        print(f"Sample image not found: {sample_img_path}")
        return
    
    image = load_image(sample_img_path)
    
    # Initialize OCR processor
    ocr = OCRProcessor(ground_truth=GROUND_TRUTH_TEXT)
    
    # Extract text
    text, cer = ocr.process_image(image)
    
    # Display results
    fig, axes = plt.subplots(1, 2, figsize=(15, 8))
    
    axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Input Image")
    axes[0].axis('off')
    
    # Display OCR results
    axes[1].axis('off')
    axes[1].text(0, 1, "OCR Results:", fontsize=14, fontweight='bold', 
                transform=axes[1].transAxes, verticalalignment='top')
    
    results_text = f"\nCharacter Error Rate: {cer:.2f}%\n\n"
    results_text += "Extracted Text:\n" + "-" * 40 + "\n"
    results_text += text[:500] + "..." if len(text) > 500 else text
    
    axes[1].text(0, 0.9, results_text, fontsize=10, fontfamily='monospace',
                transform=axes[1].transAxes, verticalalignment='top')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Ground Truth (first 200 chars):\n{GROUND_TRUTH_TEXT[:200]}...")

test_ocr()

## 7. Complete Pipeline Test

In [ ]:
# Test complete pipeline on a single image
def test_complete_pipeline():
    from src.pipeline import DocumentRestorationPipeline
    
    # Load a sample image
    sample_img_path = TRAIN_IMAGES_DIR / train_list[0]
    if not sample_img_path.exists():
        print(f"Sample image not found: {sample_img_path}")
        return
    
    # Initialize pipeline
    pipeline = DocumentRestorationPipeline(
        denoising_method="non_local_means",
        use_deskewing=True,
        use_clahe=True
    )
    
    # Process image
    results = pipeline.process_image(sample_img_path)
    
    # Display results
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    titles = ["Original", "Denoised", "Deskewed", "Final", "OCR Text", "Metrics"]
    images = [
        results.get("original_image"),
        results.get("denoised_image"),
        results.get("deskewed_image"),
        results.get("final_image")
    ]
    
    # Display images
    for i in range(4):
        if images[i] is not None:
            axes[i//3, i%3].imshow(cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB))
        axes[i//3, i%3].set_title(titles[i])
        axes[i//3, i%3].axis('off')
    
    # Display OCR text
    axes[1, 1].axis('off')
    ocr_text = results.get("extracted_text", "")
    axes[1, 1].text(0, 1, "OCR Output:", fontsize=12, fontweight='bold',
                   transform=axes[1, 1].transAxes, verticalalignment='top')
    axes[1, 1].text(0, 0.9, ocr_text[:300] + "..." if len(ocr_text) > 300 else ocr_text,
                   fontsize=9, fontfamily='monospace',
                   transform=axes[1, 1].transAxes, verticalalignment='top')
    
    # Display metrics
    axes[1, 2].axis('off')
    metrics_text = "Pipeline Metrics:\n" + "="*20 + "\n"
    metrics_text += f"CER: {results.get('cer', 0):.2f}%\n"
    metrics_text += f"PSNR: {results.get('psnr', 0):.2f} dB\n"
    metrics_text += f"SSIM: {results.get('ssim', 0):.4f}\n"
    metrics_text += f"Predicted Angle: {results.get('predicted_angle', 0):.2f}°\n"
    metrics_text += f"Total Time: {results.get('total_time', 0):.2f}s\n"
    
    axes[1, 2].text(0, 1, metrics_text, fontsize=11, fontfamily='monospace',
                   transform=axes[1, 2].transAxes, verticalalignment='top')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Complete pipeline test finished for: {sample_img_path.name}")

test_complete_pipeline()

## 8. Next Steps

To run the complete pipeline:

1. **Train the deskewing model**:
   ```bash
   python scripts/train_deskew_model.py
   ```

2. **Run the complete pipeline**:
   ```bash
   python run_pipeline.py --all
   ```

3. **Evaluate results**:
   Check the `results/` directory for output files and visualizations.